In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [5]:
from src.preprocessing import preprocess_pipeline

## Import Libraries

In [1]:
import pandas as pd
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\smrts\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\smrts\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df = pd.read_csv ("../data/train.csv", header= None)
df.columns = ["label", "title", "text"]

In [6]:
df.head()

,label,title,text
0,2,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...
1,2,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...
2,2,Amazing!,This soundtrack is my favorite music of all ti...
3,2,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...
4,2,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine..."


In [7]:
df['label'] = df['label'].map ({1:0, 2:1})

In [8]:
df["label"].value_counts()

label
1    1800000
0    1800000
Name: count, dtype: int64

## Changed the labels from (1,2) to (0,1)
Now, 
- 0 -> Negative
- 1 -> positive

In [9]:
df["text"] = df["title"] + " " + df["text"]
df = df.drop(columns=["title"])

###### “The title and review text were combined to provide richer contextual information, as titles often contain strong sentiment and summary cues. This improves both statistical feature extraction and semantic understanding, leading to better model performance.”

- Helps semantic models (LLM side)

Models like DistilBERT look at context

Combining:
- gives full context in one sequence
- avoids splitting meaning

Improves feature richness (TF-IDF side)

TF-IDF works on:

word frequency + importance

When you combine: You increase:

word diversity
important keywords
signal strength

In [10]:
### Sampling 
df = df.sample(100000, random_state=42)

In [11]:
# Clean missing values
df = df.dropna()

In [12]:
df.info()
df['label'].value_counts()
df['text'].str.len().describe()

<class 'pandas.core.frame.DataFrame'>
Index: 99993 entries, 2079998 to 100143
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   99993 non-null  int64 
 1   text    99993 non-null  object
dtypes: int64(1), object(1)
memory usage: 2.3+ MB


count    99993.000000
mean       430.306201
std        237.535779
min        100.000000
25%        231.000000
50%        381.000000
75%        593.000000
max       1014.000000
Name: text, dtype: float64

In [13]:
#### Train- Validation data set
from sklearn.model_selection import train_test_split

train_df, val_df= train_test_split(
    df,
    test_size= 0.2,
    random_state= 42,
    stratify= df["label"]
)

##### “Stratified splitting ensures that both training and validation sets maintain the same class distribution as the original dataset, preventing bias and ensuring reliable model evaluation.”

In [14]:
train_df["label"].value_counts(normalize=True)
val_df["label"].value_counts(normalize=True)
# checking stratification for biasness

label
0    0.500125
1    0.499875
Name: proportion, dtype: float64

In [15]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True) 
# easy to debug resetting random non-continuous indices to continuous indices 

In [16]:
# save train and val datasets
train_df.to_csv("../data/train_split.csv", index=False)
val_df.to_csv("../data/val_split.csv", index=False)

## Applying Text Preprocessing

In [17]:
# reading data
train_df = pd.read_csv("../data/train_split.csv")
val_df = pd.read_csv("../data/val_split.csv")
test_df = pd.read_csv("../data/test.csv", header= None)

#### Test Data Correction 

In [18]:
#1. assign column names
test_df.columns = ["label", "title", "text"] 

In [19]:
# Combine text and title and then drop title (same as train and val test)
test_df["text"] = test_df["title"] + " " + test_df["text"]
test_df = test_df.drop(columns=["title"])

In [20]:
# test
test_df.head()

,label,text
0,2,Great CD My lovely Pat has one of the GREAT vo...
1,2,One of the best game music soundtracks - for a...
2,1,Batteries died within a year ... I bought this...
3,2,"works fine, but Maha Energy is better Check ou..."
4,2,Great for the non-audiophile Reviewed quite a ...


In [21]:
# change labels here as well 
test_df["label"] = test_df["label"].map({1: 0, 2: 1})

In [33]:
#### Contd. Preprocessing 

In [22]:
# dropping missing values (bcz might give error when applying lowercase during text preprocessing)
train_df = train_df.dropna(subset=['text'])
val_df = val_df.dropna(subset=['text'])
test_df = test_df.dropna(subset=['text'])

In [23]:
# apply pre-processing 
train_df["clean_text"] = train_df["text"].apply(preprocess_pipeline)
val_df["clean_text"] = val_df["text"].apply(preprocess_pipeline)
test_df["clean_text"] = test_df["text"].apply(preprocess_pipeline)

In [24]:
train_df[["text", "clean_text"]].head()

,text,clean_text
0,"Problems Movie seemed funny, but it would have...",problem movie seemed funny would play five min...
1,Taste is an issue Yes it fits perfectly in my ...,taste issue yes fit perfectly cubicle thanks c...
2,Whirlpool 4396710 KitchenAid PUR - Water Filte...,whirlpool kitchenaid pur water filter pack shi...
3,sassy mary janes I mostly wear them to work wh...,sassy mary janes mostly wear work charge twent...
4,Its Daria This is Daria alright. (No exclamati...,daria daria alright exclamation point needed m...


In [25]:
# keeping original text for LLM 
train_df["llm_text"] = train_df["text"]
val_df["llm_text"] = val_df["text"]
test_df["llm_text"] = test_df["text"]

In [26]:
(train_df["clean_text"] == "").sum()
# checking for empty o/p

0

0 means no completely empty review after applying preprocessing meaning it wasn't very aggressive 

In [27]:
# Save pre-processed datasets
train_df.to_csv("../data/train_processed.csv", index=False)
val_df.to_csv("../data/val_processed.csv", index=False)
test_df.to_csv("../data/test_processed.csv", index=False)

In [28]:
test_df.to_csv(
    "../data/test_processed.csv",
    index=False
)

In [30]:
test_df = pd.read_csv("../data/test_processed.csv")

In [31]:
train_df = pd.read_csv("../data/train_processed.csv")
val_df = pd.read_csv("../data/val_processed.csv")
test_df = pd.read_csv("../data/test_processed.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(79994, 4)
(19999, 4)
(399976, 4)


In [32]:
print(train_df["clean_text"].isna().sum())
print(val_df["clean_text"].isna().sum())
print(test_df["clean_text"].isna().sum())

0
0
1
